In [1]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score
from imblearn.over_sampling import SMOTE
import lightgbm as lgb
from sklearn.metrics import f1_score
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import RandomizedSearchCV
from sklearn.preprocessing import OrdinalEncoder
from sklearn.compose import ColumnTransformer
import lightgbm as lgbm

In [2]:
df_trn = pd.read_csv('train.csv')
df_tst=pd.read_csv('test.csv')

In [3]:
df_trn.head()

,ID,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,label
0,train00001,34,blue-collar,married,primary,no,358,yes,no,unknown,23,may,100,4,-1,0,unknown,0
1,train00002,33,blue-collar,married,secondary,no,-53,yes,no,unknown,20,may,172,1,-1,0,unknown,0
2,train00003,32,management,single,tertiary,no,207,yes,no,cellular,2,feb,56,1,-1,0,unknown,0
3,train00004,37,blue-collar,divorced,secondary,no,638,yes,no,cellular,8,may,326,1,326,2,success,0
4,train00005,33,housemaid,married,secondary,no,826,yes,no,cellular,11,may,256,1,-1,0,unknown,0


In [4]:
df_trn.info()

<class 'pandas.core.frame.DataFrame'>
,RangeIndex: 31647 entries, 0 to 31646
,Data columns (total 18 columns):
, #   Column     Non-Null Count  Dtype 
,---  ------     --------------  ----- 
, 0   ID         31647 non-null  object
, 1   age        31647 non-null  int64 
, 2   job        31647 non-null  object
, 3   marital    31647 non-null  object
, 4   education  31647 non-null  object
, 5   default    31647 non-null  object
, 6   balance    31647 non-null  int64 
, 7   housing    31647 non-null  object
, 8   loan       31647 non-null  object
, 9   contact    31647 non-null  object
, 10  day        31647 non-null  int64 
, 11  month      31647 non-null  object
, 12  duration   31647 non-null  int64 
, 13  campaign   31647 non-null  int64 
, 14  pdays      31647 non-null  int64 
, 15  previous   31647 non-null  int64 
, 16  poutcome   31647 non-null  object
, 17  label      31647 non-null  int64 
,dtypes: int64(8), object(10)
,memory usage: 4.3+ MB


In [5]:
month_map = {
    'jan': 1, 'feb': 2, 'mar': 3, 'apr': 4, 'may': 5, 'jun': 6,
    'jul': 7, 'aug': 8, 'sep': 9, 'oct': 10, 'nov': 11, 'dec': 12
}

df_trn['month'] = df_trn['month'].map(month_map)
df_tst['month'] = df_tst['month'].map(month_map)

In [6]:
df_trn.head()

,ID,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,label
0,train00001,34,blue-collar,married,primary,no,358,yes,no,unknown,23,5,100,4,-1,0,unknown,0
1,train00002,33,blue-collar,married,secondary,no,-53,yes,no,unknown,20,5,172,1,-1,0,unknown,0
2,train00003,32,management,single,tertiary,no,207,yes,no,cellular,2,2,56,1,-1,0,unknown,0
3,train00004,37,blue-collar,divorced,secondary,no,638,yes,no,cellular,8,5,326,1,326,2,success,0
4,train00005,33,housemaid,married,secondary,no,826,yes,no,cellular,11,5,256,1,-1,0,unknown,0


In [7]:
y_cols=['label']
num_cols=['age', 'balance', 'day', 'month', 'duration', 'campaign', 'pdays', 'previous']
cat_cols=['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'poutcome']
id_cols=['ID']
x_cols=num_cols+cat_cols

In [8]:
train_corr=df_trn[num_cols].corr()
high_corr=train_corr[train_corr.abs()>=0.3]
print(high_corr)
#0.4에서 0.6이면 보통의 상관관계라 애매.

          age  balance  day  month  duration  campaign     pdays  previous
,age       1.0      NaN  NaN    NaN       NaN       NaN       NaN       NaN
,balance   NaN      1.0  NaN    NaN       NaN       NaN       NaN       NaN
,day       NaN      NaN  1.0    NaN       NaN       NaN       NaN       NaN
,month     NaN      NaN  NaN    1.0       NaN       NaN       NaN       NaN
,duration  NaN      NaN  NaN    NaN       1.0       NaN       NaN       NaN
,campaign  NaN      NaN  NaN    NaN       NaN       1.0       NaN       NaN
,pdays     NaN      NaN  NaN    NaN       NaN       NaN  1.000000  0.432882
,previous  NaN      NaN  NaN    NaN       NaN       NaN  0.432882  1.000000


In [9]:
#여러 열 labelencoding 위해 ordincalencoder사용
category_list = [['admin.', 'unknown', 'unemployed','management','housemaid', 'entrepreneur', 'student', 'blue-collar', 'self-employed', 'retired', 'technician', 'services'], 
                 ['married', 'divorced','single'], ['unknown', 'secondary', 'primary', 'tertiary'], ['yes','no'],['yes','no'],['yes','no'],
                 ['unknown', 'telephone', 'cellular'], ['unknown', 'other', 'failure', 'success']]
cat_pipeline = Pipeline([
    ('laberl_encoder',OrdinalEncoder(categories=category_list))
])

In [10]:
col_transformer = ColumnTransformer(
    [
        ('cat',cat_pipeline, cat_cols),
    ], 
    remainder='drop', 
    verbose_feature_names_out=False, 
    # force_int_remainder_cols=False
)

col_transformer.set_output(transform='pandas')

ColumnTransformer(transformers=[('cat',
                                 Pipeline(steps=[('laberl_encoder',
                                                  OrdinalEncoder(categories=[['admin.',
                                                                              'unknown',
                                                                              'unemployed',
                                                                              'management',
                                                                              'housemaid',
                                                                              'entrepreneur',
                                                                              'student',
                                                                              'blue-collar',
                                                                              'self-employed',
                                                                              'retired',
                                                                              'technician',
                                                                              'services'],
                                                                             ['married',
                                                                              'divorced',
                                                                              'single'],
                                                                             ['unknown',
                                                                              'secondary',
                                                                              'primary',
                                                                              'tertiary'],
                                                                             ['yes',
                                                                              'no'],
                                                                             ['yes',
                                                                              'no'],
                                                                             ['yes',
                                                                              'no'],
                                                                             ['unknown',
                                                                              'telephone',
                                                                              'cellular'],
                                                                             ['unknown',
                                                                              'other',
                                                                              'failure',
                                                                              'success']]))]),
                                 ['job', 'marital', 'education', 'default',
                                  'housing', 'loan', 'contact', 'poutcome'])],
                  verbose_feature_names_out=False)

In [11]:
col_transformer.fit(df_trn)

ColumnTransformer(transformers=[('cat',
                                 Pipeline(steps=[('laberl_encoder',
                                                  OrdinalEncoder(categories=[['admin.',
                                                                              'unknown',
                                                                              'unemployed',
                                                                              'management',
                                                                              'housemaid',
                                                                              'entrepreneur',
                                                                              'student',
                                                                              'blue-collar',
                                                                              'self-employed',
                                                                              'retired',
                                                                              'technician',
                                                                              'services'],
                                                                             ['married',
                                                                              'divorced',
                                                                              'single'],
                                                                             ['unknown',
                                                                              'secondary',
                                                                              'primary',
                                                                              'tertiary'],
                                                                             ['yes',
                                                                              'no'],
                                                                             ['yes',
                                                                              'no'],
                                                                             ['yes',
                                                                              'no'],
                                                                             ['unknown',
                                                                              'telephone',
                                                                              'cellular'],
                                                                             ['unknown',
                                                                              'other',
                                                                              'failure',
                                                                              'success']]))]),
                                 ['job', 'marital', 'education', 'default',
                                  'housing', 'loan', 'contact', 'poutcome'])],
                  verbose_feature_names_out=False)

In [12]:
df_trn_x = col_transformer.transform(df_trn)
df_tst_x = col_transformer.transform(df_tst)

df_trn_y = df_trn[y_cols]
df_tst_y = df_tst[y_cols]

In [13]:
smote = SMOTE(random_state=42)
df_trn_x,df_trn_y=smote.fit_resample(df_trn_x,df_trn_y)

In [14]:
n_positive = sum(df_trn_y.iloc[:, 0] == 1)  # 양성 클래스의 샘플 수
n_negative = sum(df_trn_y.iloc[:, 0] == 0)
# scale_pos_weight 설정 (양성/음성 비율)
scale_pos_weight = n_negative / n_positive

In [18]:
model = lgbm.LGBMClassifier(force_col_wise=True)

In [21]:
param_grid = {
    'n_estimators': np.arange(100, 500, 50), 
    'learning_rate': np.arange(0.01, 0.3, 0.02),
    'reg_lambda': [1, 3, 5, 10],
    'max_depth': np.arange(5,15,1),
    'subsample': np.arange(0.7, 1.0, 0.1),
    'min_split_gain': [0.01, 0.1, 0.2],
    'num_leaves': np.arange(20, 50, 5),
    'scale_pos_weight' : [scale_pos_weight]
}
#'max_depth': np.arange(3, 12, 1)

In [22]:
%%time
random_search = RandomizedSearchCV(estimator=model,
                                   param_distributions=param_grid, 
                                   scoring='f1', 
                                   cv=5, 
                                   refit=True)
random_search.fit(df_trn_x,df_trn_y.values.ravel())

[LightGBM] [Info] Number of positive: 22356, number of negative: 22356
,[LightGBM] [Info] Total Bins 1860
,[LightGBM] [Info] Number of data points in the train set: 44712, number of used features: 8
,[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
,[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
,[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
,[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
,[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
,[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
,[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
,[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
,[LightGBM] [Warning] Stopped training because there are no more leaves that meet the split requirements
,[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
,[LightG

RandomizedSearchCV(cv=5, estimator=LGBMClassifier(force_col_wise=True),
                   param_distributions={'learning_rate': array([0.01, 0.03, 0.05, 0.07, 0.09, 0.11, 0.13, 0.15, 0.17, 0.19, 0.21,
       0.23, 0.25, 0.27, 0.29]),
                                        'max_depth': array([ 5,  6,  7,  8,  9, 10, 11, 12, 13, 14]),
                                        'min_split_gain': [0.01, 0.1, 0.2],
                                        'n_estimators': array([100, 150, 200, 250, 300, 350, 400, 450]),
                                        'num_leaves': array([20, 25, 30, 35, 40, 45]),
                                        'reg_lambda': [1, 3, 5, 10],
                                        'scale_pos_weight': [1.0],
                                        'subsample': array([0.7, 0.8, 0.9, 1. ])},
                   scoring='f1')

In [25]:
best_model= random_search.best_estimator_

In [26]:
best_model.fit(df_trn_x,df_trn_y)

C:\Users\Manager\AppData\Roaming\Python\Python311\site-packages\sklearn\preprocessing\_label.py:97: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
,  y = column_or_1d(y, warn=True)
,C:\Users\Manager\AppData\Roaming\Python\Python311\site-packages\sklearn\preprocessing\_label.py:132: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
,  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


[LightGBM] [Info] Number of positive: 27945, number of negative: 27945
,[LightGBM] [Info] Total Bins 1868
,[LightGBM] [Info] Number of data points in the train set: 55890, number of used features: 8
,[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000


LGBMClassifier(force_col_wise=True, learning_rate=0.12999999999999998,
               max_depth=13, min_split_gain=0.01, n_estimators=200,
               num_leaves=40, reg_lambda=3, scale_pos_weight=1.0,
               subsample=0.9999999999999999)

In [27]:
y_pred = best_model.predict(df_tst_x)

In [28]:
df_submission = pd.read_csv('submission_example.csv')

In [29]:
df_submission['label'] = y_pred

In [30]:
(df_submission['label']==1).sum()

3682

In [31]:
filename = 'smote+light.csv'
df_submission.to_csv(filename, index=False)